# 2. Model Quantization

In this notebook, we perform the following steps:
- Load the initial checkpoint (stored in "../checkpoints") in float16 mode.
- Move the model to CPU to perform dynamic quantization on its Linear layers.
- Save the quantized model checkpoint for subsequent pipeline steps.

**Note:** Dynamic quantization is performed on the CPU. This notebook adheres to our strict constraints,
ensuring that all operations use the lowest precision (float16 / int8) to keep the memory footprint minimal.


In [1]:
import os
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

# Paths for input checkpoint and output quantized checkpoint
INPUT_CHECKPOINT_PATH = "../checkpoints"
OUTPUT_CHECKPOINT_DIR = "../checkpoints/model_quantized"
os.makedirs(OUTPUT_CHECKPOINT_DIR, exist_ok=True)


## 1. Load Initial Model and Tokenizer

Load the model and tokenizer in float16 mode.


In [2]:
print("Loading the initial model and tokenizer in float16 mode...")
try:
    model = AutoModelForCausalLM.from_pretrained(
        INPUT_CHECKPOINT_PATH,
        torch_dtype=torch.float16,  # Load in lower precision to reduce memory footprint
        low_cpu_mem_usage=True      # Further optimize memory usage during loading
    )
    tokenizer = AutoTokenizer.from_pretrained(INPUT_CHECKPOINT_PATH)
    print("Model and tokenizer loaded successfully.")
except Exception as e:
    print(f"Error loading model/tokenizer: {e}")
    raise

Loading the initial model and tokenizer in float16 mode...
Model and tokenizer loaded successfully.


## 2. Move Model to CPU for Quantization

Dynamic quantization in PyTorch is currently supported only on the CPU. We move the model to CPU before applying quantization.


In [3]:
try:
    print("Moving model to CPU for quantization...")
    model.to("cpu")
except Exception as e:
    print(f"Error moving model to CPU: {e}")
    raise

Moving model to CPU for quantization...


## 3. Apply Dynamic Quantization

We apply dynamic quantization targeting all the `torch.nn.Linear` layers. This reduces the model size by converting weights to int8.


In [4]:
try:
    print("Applying dynamic quantization to model's Linear layers...")
    quantized_model = torch.quantization.quantize_dynamic(
        model,
        {torch.nn.Linear},  # Specify which layers to quantize dynamically
        dtype=torch.qint8   # Target lower precision quantization (int8)
    )
    print("Dynamic quantization applied successfully.")
except Exception as e:
    print(f"Error during quantization: {e}")
    raise

Applying dynamic quantization to model's Linear layers...
Dynamic quantization applied successfully.


## 4. Save the Quantized Checkpoint

Save the quantized model and tokenizer so that subsequent notebooks can use this checkpoint.


In [6]:

print("Saving the quantized model checkpoint...")

try:
    # Get state dict from the quantized model
    state_dict = quantized_model.state_dict()
    # Filter state dict to retain only tensor objects
    filtered_state_dict = {k: v for k, v in state_dict.items() if isinstance(v, torch.Tensor)}
    
    # Define the path for the saved state dict
    state_dict_path = os.path.join(OUTPUT_CHECKPOINT_DIR, "pytorch_model.bin")
    torch.save(filtered_state_dict, state_dict_path)
    print(f"Quantized model state dict saved successfully in '{state_dict_path}'.")

    # Save the model configuration and tokenizer
    quantized_model.config.to_json_file(os.path.join(OUTPUT_CHECKPOINT_DIR, "config.json"))
    tokenizer.save_pretrained(OUTPUT_CHECKPOINT_DIR)
    print("Model configuration and tokenizer saved successfully.")
except Exception as e:
    print(f"Error saving quantized checkpoint: {e}")
    raise

Saving the quantized model checkpoint...
Quantized model state dict saved successfully in '../checkpoints/model_quantized/pytorch_model.bin'.
Model configuration and tokenizer saved successfully.


## Completion Message

The 2_quantization notebook has successfully applied dynamic quantization and saved the checkpoint.


In [7]:
print("2_quantization notebook execution complete. Proceed with further optimizations in the pipeline.")

2_quantization notebook execution complete. Proceed with further optimizations in the pipeline.
